# MIDOG++ count-error region selection - Colab runner

Runs the pipeline defined in this repository on a CUDA runtime. The local machine (Windows,
no GPU) is used for audit, tests and the interface only; see `docs/COMPUTE.md`.

**Before running:** this notebook expects the project code in your Google Drive at
`MyDrive/midogcx_project` and, optionally, specimen images at `MyDrive/midogpp_data`.
The 55-specimen cohort is 7.52 GiB.

Specimens are staged once per session to `/content/midogpp_data` and read from there:
a region crop still decodes the whole TIFF, so reading from the Drive mount would cost
hours of I/O per training run. Drive holds the dataset as cold storage only.

**Runtime > Run all** works end to end on either device. It picks CUDA when present and
falls back to CPU with a printed cost warning, reuses an existing baseline checkpoint
instead of retraining it, and reuses a finished baseline gate. Expected GPU time: ~1.5 h
(gate ~10 min, five branches ~60 min). Expected CPU time: ~7-8 h.

This notebook produces real experimental output. It does not fabricate or approximate any
number: if a step cannot run, it stops and says why.

### Fixed in this version (notebook driver only; the package logic is unchanged)

* `--set` overrides are passed as **separate argv items**. The previous version joined
  them into one string, so `training.device` became the literal value
  `"cuda training.batch_size=4 training.updates=500"` and torch raised
  `RuntimeError: Invalid device string` on every GPU run; `batch_size` was also silently
  left at 2.
* Dependencies are installed **only when missing**, so pip cannot replace Colab's CUDA
  torch build or its working OpenCV with a headless wheel.
* `pytest` is installed when absent (not guaranteed in a Colab image).
* Training retries once at batch 2 on CUDA out-of-memory; checkpoint selection walks
  newest-to-oldest and skips a torn file instead of aborting.
* A preflight names a missing project file before any GPU time is spent.
* The archive cell copies small result artefacts only (checkpoints are regenerable) and
  no longer depends on a variable defined by another cell.


In [ ]:
import shutil, subprocess, sys

smi = shutil.which("nvidia-smi")
if smi:
    out = subprocess.run(
        [smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    print(out.stdout.strip() or out.stderr.strip())
else:
    print("nvidia-smi not found: this runtime has no GPU attached.")
    print("Runtime > Change runtime type > T4 GPU, then re-run this cell and the next.")
print("python", sys.version.split()[0])


In [ ]:
# Colab already ships torch/torchvision (CUDA build) plus numpy, pandas, scipy,
# matplotlib and scikit-learn. Install only what is genuinely missing: letting pip
# reinstall torch or swap Colab's OpenCV for a headless wheel is the usual way this
# pipeline breaks before training starts.
import importlib, subprocess, sys


def _missing(module):
    try:
        importlib.import_module(module)
        return False
    except Exception:
        return True


need = []
for module, package in [
    ("cv2", "opencv-python-headless"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
    ("pandas", "pandas"),
    ("tqdm", "tqdm"),
    ("pytest", "pytest"),  # not guaranteed to be in the Colab image
]:
    if _missing(module):
        need.append(package)

if need:
    print("installing missing packages:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
else:
    print("all pipeline dependencies are already present")

import torch, torchvision

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())

# Run all works on either device. Measured cost: 5.85 s/update on this project's CPU
# against 0.76 s/update on a T4, i.e. ~7 h for the five branches instead of ~30 min.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4 if DEVICE == "cuda" else 2
if DEVICE == "cuda":
    print("using:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no CUDA device - training will run on CPU.")
    print("  ~50 min per 500-update run, ~7-8 h for the whole notebook.")
    print("  For ~1.5 h instead: Runtime > Change runtime type > T4 GPU, restart, Run all.")
print("DEVICE =", DEVICE, "| BATCH_SIZE =", BATCH_SIZE)


## 1. Project code and paths

If you have not uploaded the project yet, upload the folder as a zip to MyDrive, then set
`PROJECT_DIR` accordingly. The notebook never edits the project; it only runs its scripts.


In [ ]:
import glob, hashlib, os, shutil, subprocess, sys

PROJECT_DIR = "/content/drive/MyDrive/midogcx_project"
DRIVE_DATA = "/content/drive/MyDrive/midogpp_data"  # 7.52 GiB, cold storage only
LOCAL_DATA = "/content/midogpp_data"  # session disk, used for reading

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    # Not Colab: take the paths from the environment so the notebook still runs.
    PROJECT_DIR = os.environ.get("PROJECT_DIR", os.getcwd())
    DRIVE_DATA = os.environ.get("MIDOG_DATA_ROOT", DRIVE_DATA)
    LOCAL_DATA = DRIVE_DATA
    print("not running in Colab; PROJECT_DIR =", PROJECT_DIR)


def _n_tiff(root):
    return len(glob.glob(os.path.join(root, "*.tiff")))


DATA_ROOT = LOCAL_DATA
os.makedirs(LOCAL_DATA, exist_ok=True)
if os.path.abspath(LOCAL_DATA) != os.path.abspath(DRIVE_DATA) and _n_tiff(DRIVE_DATA) > _n_tiff(LOCAL_DATA):
    print("staging specimens from Drive to local disk (7.52 GiB, once per session) ...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print("staged:", _n_tiff(LOCAL_DATA), "specimens")

os.environ["MIDOG_DATA_ROOT"] = DATA_ROOT
if not os.path.isdir(PROJECT_DIR):
    raise SystemExit(f"project not found at {PROJECT_DIR}: clone or upload the repo, then set PROJECT_DIR")
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# The mounted copy can lag this repository, and a stale copy has already cost a baseline
# run: an older midogcx/models/error_predictor.py let the config's n_splits reach
# HistGradientBoostingRegressor, which failed only after the source detector had trained.
# So check every file the run depends on here, before any GPU time is spent. The manifest is
# the sha256 of each file; regenerate it after changing the repository with
#     python scripts/write_notebook_manifest.py
# Deliberately absent: docs/ and the top-level *.md (prose the run never reads, so a doc edit
# must not invalidate the manifest), notebooks/*.ipynb (Colab autosaves execution state into
# the mounted copy, so its bytes change during a session), data/metadata/figshare_cache/ (raw
# download cache, read by no step) and .git, __pycache__, *.pyc.
MANIFEST = """
ff92fe30696122a65b86ec586fb8091146e99e0901183b196d39cfcc0d4e1787  configs/base.yaml
19fb5ef005cea33469497ee68a713159b7875d9f64a741afec7719bc9917eb6d  configs/cohort_v1.yaml
8c87ab4276eb11f6b6b5ac0d956b89dad7568a5474e0904b653d54dc68091f47  data/metadata/MIDOG++.json
a3b38228b9e4915bf92ff4010d4edfea1cf83fec8df4f4666f3b3e8aa75c965d  data/metadata/audit.json
749b3cb475c8566cbd108d764860a22fa10c3094017c205a5b128485972cbb8c  data/metadata/dataset_inventory.csv
4a8a14739b0aeefcff597b9d12e519a58d155cb162722bb5e482bc48dc588e53  data/metadata/dataset_inventory_summary.json
211140f590199d50ee98d923ddc923c382c3a5543ea3b430612c3624570b26b8  data/metadata/datasets_xvalidation.csv
798d6ec2b6431d3fd1622feb478784c004b6f95625fdd742bb902acb37147ba0  data/metadata/figshare_collection.json
7500ae4a2fd9d1fd8e249893bdc0552e5daac0abf4a154f3315726d7987a18d5  data/metadata/manifest_with_sizes.csv
a748761597233857f2903e08ef2bf79ccfe80c347a451c306b0efde6e2b65bcb  data/metadata/official/configs_all.yaml
a01e43b79ba3f366061f8fa8d92223c7e30298d4e1e0c1691bbd301ef7533a1c  data/metadata/official/evaluation.py
b5f74e68ece208e56be5baced0808e2cca4adbf22b795c16540a9d4fafcb1255  data/metadata/official/inference.py
338781a07c0d815bd4d94d34b3b71cf5a991ad6f378739f3bbe9bb1e67a12c27  data/metadata/official/requirements.txt
2ce2334489f3b3ab7e1ae071c44ba3eadb365d8b33ab94d5374576ebee6544a9  data/metadata/official/training.py
3c4280729c4757c355236de90883dd06ba7d56d9dea330f71c90b659e3e62c9f  data/metadata/official/utils_nms_WSI.py
c2890ce50048ddd7749eeb6652876cacda2228806f088f7d04843c3b0f81d2aa  data/splits/cohort_v1.json
d58368ec4f6b27b1f95c2dba1b4fa114902d7fecc6e2e1e619035a8e5944f297  data/splits/cohort_v1_summary.json
b3b224dbbff059dd6b5c38aa2f6870a348437fd97f363bd4e1f1bcceb22bdab1  data/splits/geometry.csv
6c145097b31666bf59e4edb88cbfc1bdf8e3d1c6cf248a109fcd19a9de4b0dd5  data/splits/regions.csv
c44fe75200fc2947149893c1f17b3889ec61240dca555e0f49aa513b603669d5  frontend/app.py
59cb87727c2645c510cacf3bfb7a398af9df8c75a1e4d32f1d5b458ec472141f  midogcx/__init__.py
282eed3be1f49510d173b7943fb43a4a1fb222d3967d99b4c2adbe508713448c  midogcx/acquisition.py
674f4b7744354a12d1351b56f890983584a3fb82af9f2f1ac390303afa1194bf  midogcx/adaptation/__init__.py
fbb8ca2daf44d3547b1e06adf8bc4be3b4beed33ee2300be95d009e6457b7384  midogcx/adaptation/trainer.py
a9ebc74f5a36d0b169e70d4a5cbade0bbff22c468977e3fea54e58fbb3dc1d71  midogcx/config.py
5eafa27f017013d5df95359671885b975990b33da76baaf7d789c32746ee1bd4  midogcx/data/__init__.py
6386984f28d0b27524b542b3db8464a8f4e2c67a4764860460acbac55d739446  midogcx/data/dataset.py
53db168bb0a77d8c7987e02c12e53ee481a50a4f846d6d334e513add76ebe550  midogcx/data/images.py
d5d4b74f7cf73896ada82293fc2b32afa1a97e14415f6a3a100373f717a67f4f  midogcx/data/metadata.py
7c63edfa8fa324057ef2acfabc3cf6db88fce3d296c8342483e604dd32b77911  midogcx/data/patches.py
1a21b067cb1d5b4eb2a1f10ec9018c69bd98e1c0684428f73a42fd7d5af2a063  midogcx/data/splits.py
5e5a376352c42a2d5cd28a8d9662b68c22ba7ee9efcd686a604f24745415f00b  midogcx/evaluation/__init__.py
354da8537bed7dba320401b4ab2d3c729198e3439697925bfa4d98103569773e  midogcx/evaluation/evaluate.py
3c1b779711ac94699f8085c5328cf04993df50b324992bea80a39dc3d80558c0  midogcx/evaluation/matching.py
acb6655478a99c81a68984722d240681d927673d008903307544b006e1a932d0  midogcx/evaluation/metrics.py
b91843c37e1cb1e0a4140ed2cdd924d7eb76a109044c12da3c3718a06f3e7f21  midogcx/evaluation/nms.py
e70184b623cf5627dbd1a962e58f649685b34710e77ce4ea33d06f593192ed2e  midogcx/inference/__init__.py
ee036813c67f8963dc7becc96f78b7150f3a1aa25cda7c44b2d06ba4e2e0406b  midogcx/inference/sliding_window.py
fe6d3ddccb3476783b2cf42cdb5cbbf37ebb4de273e2d324b22ad1e249bccb85  midogcx/models/__init__.py
62bb2868b1248963a04391b698c86902cadd5ef72f5fd292d9798cfe4c4b30aa  midogcx/models/detector.py
299bcfc44e442239133d9a982410c82dd1d5628fcf5a28c2e1a28309bc61264a  midogcx/models/error_predictor.py
9737e57ee22d23af4f2f4d330ae900af8085aa0fc96068cefeca83145486aa1e  midogcx/selection/__init__.py
6f3250aa2367c9c9da2ab44bb5b91f21b4508d2fde3bcdc7b4dca2b2f40d5204  midogcx/selection/budget.py
34b779977bfcdaadb35e65933c1ced495dd9f8796d5c20cae59432788eb1c000  midogcx/selection/strategies.py
6afdc34dc5ca08049da2736e7e87009d1cf6d06ca64e53c78347014a323a8c9c  midogcx/utils.py
1ec23de1e5f3dc524b3ea28a5516fdb1b1d4f9bd4899160f153f204578f5b982  outputs/metadata/normalization.json
4f129828e62bb8f9c8c6210df851675b7a29dd15af03ccef4ddcc0383c8396be  requirements.txt
c16ab475c7c721df47bf6fb1275e02a151b61c43d2c890388a4a9be2d53bcf9a  scripts/audit_dataset.py
f9e701aad6be1ede4b70884e9aaf80099f78fac7e194903bda2264b04f6f10ce  scripts/audit_dataset_inventory.py
07b995d6393e31178b47bf24323e6f32dce5de4f627b7b3afe7d22b2bb78c841  scripts/build_cohort.py
301cda02da239662da69eaf36dfeceb6148c298d9f38869b397e85825047f67c  scripts/download_specimens.py
b28844fa48d4f03471a39b0316f2adfcae8591ef90fce29a2ffa947b16f3b11b  scripts/infer_predictions.py
0e69c28f2078e0ac25a755b6053c868dc4d07d1e32c8b9d1d6b3c977a43d9c63  scripts/prepare_regions.py
9c4dcf97d3de90d8306817c2bde588762d1628ecd3ecb9907cc3e80600770e3c  scripts/run_acquisition_experiment.py
6ad53706978b8e27c95400c86e3733c6c752f69f7735cdc0e02f3022266410eb  scripts/smoke_test.py
5d6a1f4657810dfd24ed0e0dfeb45f79fc19aabeb447e4234a411ee3f144670c  scripts/train_detector.py
67179500ff2acf03c138292554f723448ddf47bf06b88d64b8218f4f59aca2d4  scripts/verify_checkpoint.py
68ec62346e209c818f46b0caab8098aae5a3a4975f0ae324b77376d04cd86383  scripts/verify_readiness.py
438eb0831dff7d75389186473f8b55359adf97144bec56bb9367cb854b9ba4a4  scripts/write_notebook_manifest.py
ac501221e41ba9ec8d22d816b6c2ade8efe9039593c1b36d1e6a02fbcf089fa9  tests/conftest.py
7e413799e3f3dc3b325212936f790f696489b2607171e206ce8ca736b5cf2940  tests/test_dataset.py
1d5616f6b34b4f1b552c3d32565012b0ba120f99bd4a5c411329d41bd3518891  tests/test_detector.py
27f9adf97d28209e18af489686d8c461f328c99483840296e614685d316fb41c  tests/test_error_predictor.py
ed47a9023abc592e2dd9ed7fc14e39ba2dc87b91ba35c57060bd3dd07054fd4b  tests/test_evaluate.py
4247206f3c22cc67838104f6ad3444f13bd81898350b59f5a12ec60a4ed7119e  tests/test_matching.py
d81f0f2a18df40fa842eacd7783e00f9daa19ebe381420ef9998418022a5ee07  tests/test_metadata.py
c0d31df56bf69d3b42bc0aee15bceab11c0e77ce7aa75e688bde5780dbf1341e  tests/test_metrics.py
df44a61a6621714af85aa292ed17b27563c1f3b9c78a877aa88d0b824d9eaa2e  tests/test_nms.py
b04bf5b2ceaf889567db767d9725dc5facfdecba813426d2d1e49f75c760c3e3  tests/test_patches.py
146d09b63425185516329ab4322c8135d97dc3b5111b25709ae372187b035494  tests/test_readiness.py
eff73d13764866c7be7625adceec5c96e1219e4b9646194550457e896d28563a  tests/test_selection.py
b97a0f7c6819a750d9291111124f5c639efc82e3418e681b6da52212ae84af95  tests/test_splits.py
"""

_expected = {}
for _line in MANIFEST.strip().splitlines():
    _digest, _rel = _line.split("  ", 1)
    _expected[_rel] = _digest


def _sha256(path):
    _h = hashlib.sha256()
    with open(path, "rb") as _fh:
        for _chunk in iter(lambda: _fh.read(1 << 20), b""):
            _h.update(_chunk)
    return _h.hexdigest()


# Files this copy has and the repository does not: a partial copy, or leftovers of an older
# one. Same skip rules as scripts/write_notebook_manifest.py.
_SKIP_DIRS = {"__pycache__", "figshare_cache"}
_SKIP_SUFFIXES = {".pyc", ".pyo", ".ipynb"}
_present = set()
for _root in ("midogcx", "scripts", "configs", "tests", "frontend", "data"):
    for _dirpath, _dirnames, _filenames in os.walk(_root):
        _dirnames[:] = [d for d in _dirnames if d not in _SKIP_DIRS and not d.startswith(".")]
        for _name in _filenames:
            if _name.startswith(".") or os.path.splitext(_name)[1] in _SKIP_SUFFIXES:
                continue
            _present.add(os.path.relpath(os.path.join(_dirpath, _name)).replace(os.sep, "/"))

_missing = sorted(p for p in _expected if not os.path.exists(p))
_changed = sorted(p for p in _expected if p not in _missing and _sha256(p) != _expected[p])
_extra = sorted(_present - set(_expected))

if _missing or _changed or _extra:
    print(f"the project copy at {PROJECT_DIR} does not match this repository:")
    for _label, _files in (("changed", _changed), ("missing", _missing), ("not in the repository", _extra)):
        if _files:
            _shown = ", ".join(_files[:10]) + (f", ... ({len(_files)} in total)" if len(_files) > 10 else "")
            print(f"  {_label}: {_shown}")
    raise SystemExit(
        "stale or partial project copy. If the repository itself changed, regenerate the manifest "
        "first (python scripts/write_notebook_manifest.py); then re-upload the repository to "
        "PROJECT_DIR and re-run this cell. Do not spend training time on this copy."
    )

print(f"project copy verified: {len(_expected)} files match this repository")

print("project:", os.getcwd())
print("specimens:", DATA_ROOT, "->", _n_tiff(DATA_ROOT), "tiff")
print(
    "free space (GB) - local:",
    round(shutil.disk_usage(LOCAL_DATA).free / 1e9, 1),
    "| results disk:",
    round(shutil.disk_usage(PROJECT_DIR).free / 1e9, 1),
)


def run_step(*argv, allow_failure=False):
    """Run a project script in this interpreter and surface the script's own error.

    Each argument is passed as its own argv item. Never join several ``--set key=value``
    tokens into one string: argparse's ``nargs="*"`` would receive a single value
    containing spaces, so ``training.device`` became the literal string
    "cuda training.batch_size=4 training.updates=500" and torch raised
    ``RuntimeError: Invalid device string``.

    stdout is inherited so progress streams (the child runs with PYTHONUNBUFFERED);
    stderr is captured so a failure prints the child's own traceback verbatim.
    """
    argv = [str(a) for a in argv]
    print(f"\n>>> {sys.executable} " + " ".join(argv), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.run([sys.executable, *argv], text=True, stderr=subprocess.PIPE, env=env)
    if proc.returncode != 0:
        print(f"\n!!! step failed (exit {proc.returncode}): {' '.join(argv)}")
        print(proc.stderr or "(the script said nothing on stderr; read the lines printed above)")
        if allow_failure:
            return proc
        raise SystemExit("fix the error printed above, then re-run this cell")
    return proc


def train_command(batch, updates, tag="source_baseline", extra=()):
    """`train_detector.py` argv with device and batch forced to this runtime's choice."""
    return [
        "scripts/train_detector.py",
        "--tag",
        tag,
        "--set",
        f"training.device={DEVICE}",
        f"training.batch_size={batch}",
        f"training.updates={updates}",
        *extra,
    ]


In [ ]:
# Sanity: code must pass its own tests before any training time is spent.
run_step("-m", "pytest", "tests/", "-q")


## 2. Dataset, frozen cohort and frozen region grid

Metadata ships with the project. Both frozen artefacts are **verified, never
regenerated**: the split and the region grid define every selection, budget and
previously reported number, so a remote runtime must not be able to rewrite them.


In [ ]:
# Verifies the frozen cohort instead of rewriting it: the split is immutable.
run_step("scripts/build_cohort.py", "--check")


In [ ]:
# Exact download sizes first. Nothing is fetched by --dry-run.
run_step("scripts/download_specimens.py", "--dry-run")


In [ ]:
# The cohort is 7.52 GiB. The downloader refuses to start if the disk cannot hold it,
# resumes partial files, and verifies byte-exact sizes. Files already staged from Drive
# are size-checked and skipped.
run_step("scripts/download_specimens.py")


## 3. Regions and normalisation statistics

Reads every cohort specimen once: writes the non-overlapping 512x512 region grid, the
physical spacing table, and channel statistics computed **only** from source training
specimens.


In [ ]:
# Verifies the frozen region grid (regions.csv, geometry.csv, normalization.json)
# against a fresh recomputation and writes nothing: selections and budgets are
# defined against this exact grid.
run_step("scripts/prepare_regions.py", "--check")


## 4. Source-domain detector

A real run needs far more than the smoke-test budget. Measure first: the script logs
seconds per update, so a session can be cut short without losing the result. The checkpoint
stores its own architecture, normalisation and update count.


In [ ]:
import glob, os

RETRAIN_BASELINE = False  # True forces a fresh baseline even if a checkpoint exists
BASELINE_UPDATES = 500

existing = sorted(glob.glob("outputs/runs/*/checkpoint.pth"), key=os.path.getmtime, reverse=True)
if existing and not RETRAIN_BASELINE:
    print("reusing existing baseline checkpoint:", existing[0])
    print("set RETRAIN_BASELINE = True in this cell to train a new one")
else:
    print("training the source baseline on", DEVICE, "at batch", BATCH_SIZE)
    proc = run_step(*train_command(BATCH_SIZE, BASELINE_UPDATES), allow_failure=True)
    if proc.returncode != 0:
        # A T4 at batch 4 sits close to its memory limit; one retry at batch 2 is cheaper
        # than losing the session. The failed attempt leaves a run directory but no
        # checkpoint, so the selector below cannot pick it up.
        if "out of memory" in (proc.stderr or "").lower() and BATCH_SIZE > 1:
            print(f"\nCUDA out of memory at batch {BATCH_SIZE}: retrying once at batch 2")
            run_step(*train_command(2, BASELINE_UPDATES))
        else:
            raise SystemExit("training failed: read the error printed above")


In [ ]:
import glob, os, torch

ckpts = sorted(glob.glob("outputs/runs/*/checkpoint.pth"), key=os.path.getmtime, reverse=True)
print("\n".join(ckpts[:3]) or "(no checkpoints found)")
if not ckpts:
    raise SystemExit("no checkpoint under outputs/runs/*/checkpoint.pth - run the training cell first")

# Newest first: a run interrupted while saving leaves a file that exists but cannot be
# read. Fall through to the next-newest readable checkpoint, printing each skipped file.
SOURCE_CKPT = None
for candidate in ckpts:
    try:
        torch.load(candidate, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"skipping unreadable checkpoint {candidate} ({exc})")
        continue
    SOURCE_CKPT = candidate
    break
if SOURCE_CKPT is None:
    raise SystemExit("every checkpoint under outputs/runs is unreadable; set RETRAIN_BASELINE = True")
print("using:", SOURCE_CKPT)


## 5. Acquisition experiment

Threshold calibration, predictor fitting on out-of-sample source specimens, the five
rankings, matched adaptation branches and independent melanoma assessment. A `source_only`
control is always evaluated.


In [ ]:
# GATE: evaluate the unadapted control before spending time on any branch.
# If precision/recall/F1 are near zero here, the acquisition comparison is
# meaningless, so fix the baseline first.
import glob, shutil

gate_done = glob.glob("outputs/experiments/*baseline_gate/metrics/source_only.json")
print("free space where results are written (GB):", round(shutil.disk_usage(".").free / 1e9, 1))
if gate_done:
    print("baseline gate already evaluated:", gate_done[-1])
    print("delete that run directory to evaluate it again")
else:
    run_step(
        "scripts/run_acquisition_experiment.py",
        "--checkpoint", SOURCE_CKPT,
        "--tag", "baseline_gate",
        "--device", DEVICE,
        "--stop-after-control",
    )


In [ ]:
# Five matched adaptation branches plus the source_only control, all on one device.
run_step(
    "scripts/run_acquisition_experiment.py",
    "--checkpoint", SOURCE_CKPT,
    "--tag", "pilot",
    "--device", DEVICE,
)


In [ ]:
import glob, json, os

import pandas as pd

runs = [
    r
    for r in sorted(glob.glob("outputs/experiments/*"), key=os.path.getmtime)
    if os.path.exists(os.path.join(r, "comparison.csv"))
]
if not runs:
    raise SystemExit("no completed run under outputs/experiments - run the gate/full experiment first")
run = runs[-1]
print(run)
print(pd.read_csv(os.path.join(run, "comparison.csv")).to_string(index=False))
report = os.path.join(run, "predictor_report.json")
if os.path.exists(report):
    with open(report) as fh:
        print(json.dumps(json.load(fh), indent=2)[:4000])


## 6. Second seed (optional, needed for a variability estimate)

Everything is seeded from `project.seed`; changing it produces an independent replicate.
Report seed variability separately from specimen variability.


In [ ]:
# Independent replicate (report seed variability separately from specimen variability).
# Off by default: it doubles GPU time. Uses the same argv construction as cell 14.
RUN_SECOND_SEED = False
SEED2 = 7

if RUN_SECOND_SEED:
    run_step(
        *train_command(BATCH_SIZE, BASELINE_UPDATES, tag="source_seed2", extra=(f"project.seed={SEED2}",))
    )
    seed2_ckpts = sorted(
        glob.glob("outputs/runs/*source_seed2*/checkpoint.pth"), key=os.path.getmtime, reverse=True
    )
    if not seed2_ckpts:
        raise SystemExit("no checkpoint for the second seed; training must have failed above")
    run_step(
        "scripts/run_acquisition_experiment.py",
        "--checkpoint", seed2_ckpts[0],
        "--tag", "pilot_seed2",
        "--device", DEVICE,
        "--set", f"project.seed={SEED2}",
    )
else:
    print("second seed skipped (set RUN_SECOND_SEED = True to run it)")


## 7. Persist results

Small result files are what you actually need to keep; checkpoints can be regenerated.


In [ ]:
import glob, os, shutil

runs = [
    r
    for r in sorted(glob.glob("outputs/experiments/*"), key=os.path.getmtime)
    if os.path.exists(os.path.join(r, "comparison.csv"))
]
if not runs:
    raise SystemExit("no completed experiment under outputs/experiments - run the gate first")
run = runs[-1]

ARCHIVE = "/content/drive/MyDrive/midogcx_results"
os.makedirs(ARCHIVE, exist_ok=True)
dest = os.path.join(ARCHIVE, os.path.basename(run))
# Copy the small artefacts (JSON/CSV/predictor pickles) and skip checkpoints: they are
# ~150 MB each, regenerable from config + seed, and would turn a Drive write into a long
# upload for no benefit.
shutil.copytree(run, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pth", "checkpoint*"))
print("archived:", dest)
for name in sorted(os.listdir(dest)):
    print("  ", name)

KEEP_CHECKPOINTS = False  # True also copies the source baseline checkpoint (~150 MB)
if KEEP_CHECKPOINTS and "SOURCE_CKPT" in globals():
    target = os.path.join(ARCHIVE, os.path.basename(os.path.dirname(SOURCE_CKPT)) + "_checkpoint.pth")
    shutil.copy2(SOURCE_CKPT, target)
    print("copied checkpoint:", target)


## What to report

Use `comparison.csv` for the method table and `paired_comparisons.json` for per-specimen
differences with intervals. Always report counting error **together with** precision, recall,
F1 and the raw FP/FN counts - count error can be small while false positives and false
negatives cancel. Do not present any pilot-era number from the research blueprint as a result
of this run.
